# 제12장(b): 생성 AI 거버넌스

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch12b.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently pandas scikit-learn matplotlib seaborn numpy

### 생성형 AI 거버넌스

## 생성형 AI의 고유 리스크

### 환각(Hallucination)과 사실성

### 저작권과 지적재산권

### 딥페이크와 허위정보

### 프롬프트 인젝션과 보안

**프롬프트 인젝션 방어 시스템**

In [ ]:
import re
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from enum import Enum
from datetime import datetime

class ThreatLevel(Enum):
    SAFE = "safe"
    SUSPICIOUS = "suspicious"
    BLOCKED = "blocked"

@dataclass
class DefenseResult:
    level: ThreatLevel
    original_input: str
    sanitized_input: Optional[str]
    threats_detected: List[str]
    timestamp: str

class PromptInjectionDefenseSystem:
    """Multi-layered prompt injection defense"""

    INJECTION_PATTERNS = [
        r"(?i)ignore\s+(all\s+)?(previous|above|prior)\s+"
        r"(instructions|prompts|rules)",
        r"(?i)you\s+are\s+now\s+(?:a|an|acting\s+as)",
        r"(?i)system\s*:\s*",
        r"(?i)forget\s+(everything|all|your\s+instructions)",
        r"(?i)do\s+not\s+follow\s+(your|the)\s+rules",
        r"(?i)override\s+(safety|content|ethical)\s+"
        r"(filters?|guidelines?|policies?)",
        r"(?i)\[INST\]|\[\/INST\]|<<SYS>>|<\|im_start\|>",
    ]

    UNICODE_BYPASS_PATTERNS = [
        r"[\u200b-\u200f]",  # Zero-width characters
        r"[\u2028-\u2029]",  # Line/paragraph separators
        r"[\ufeff]",         # Byte order mark
        r"[\u00ad]",         # Soft hyphen
    ]

    def __init__(self, max_input_length: int = 4000,
                 system_prompt_hash: str = ""):
        self.max_input_length = max_input_length
        self.system_prompt_hash = system_prompt_hash
        self.audit_log: List[DefenseResult] = []
        self._compiled_patterns = [
            re.compile(p) for p in self.INJECTION_PATTERNS
        ]
        self._unicode_patterns = [
            re.compile(p) for p in self.UNICODE_BYPASS_PATTERNS
        ]

    def validate_input(self, user_input: str) -> DefenseResult:
        """Layer 1: Input validation and sanitization"""
        threats = []

        # Check input length
        if len(user_input) > self.max_input_length:
            threats.append(
                f"Input exceeds max length: "
                f"{len(user_input)} > {self.max_input_length}"
            )

        # Check for injection patterns
        for pattern in self._compiled_patterns:
            if pattern.search(user_input):
                threats.append(
                    f"Injection pattern detected: "
                    f"{pattern.pattern[:50]}..."
                )

        # Check for unicode bypass attempts
        for pattern in self._unicode_patterns:
            if pattern.search(user_input):
                threats.append("Unicode bypass attempt detected")

        # Determine threat level
        if len(threats) >= 2:
            level = ThreatLevel.BLOCKED
        elif len(threats) == 1:
            level = ThreatLevel.SUSPICIOUS
        else:
            level = ThreatLevel.SAFE

        # Sanitize input
        sanitized = self._sanitize(user_input) if threats else user_input

        result = DefenseResult(
            level=level,
            original_input=user_input[:200],
            sanitized_input=sanitized[:200] if sanitized else None,
            threats_detected=threats,
            timestamp=datetime.now().isoformat(),
        )
        self.audit_log.append(result)
        return result

    def validate_output(self, output: str,
                        system_prompt: str) -> Tuple[bool, List[str]]:
        """Layer 3: Output validation"""
        issues = []

        # Check for system prompt leakage
        prompt_fragments = system_prompt.split()
        consecutive_matches = 0
        for word in prompt_fragments:
            if word in output:
                consecutive_matches += 1
            else:
                consecutive_matches = 0
            if consecutive_matches >= 5:
                issues.append("Potential system prompt leakage")
                break

        # Check for sensitive data patterns
        sensitive_patterns = {
            "API Key": r"(?i)(api[_-]?key|secret[_-]?key)\s*[:=]\s*\S+",
            "Internal URL": r"https?://internal\.|https?://10\.",
            "Credentials": r"(?i)(password|passwd|pwd)\s*[:=]\s*\S+",
        }
        for name, pattern in sensitive_patterns.items():
            if re.search(pattern, output):
                issues.append(f"Sensitive data detected: {name}")

        return len(issues) == 0, issues

    def _sanitize(self, text: str) -> str:
        """Remove suspicious characters and normalize"""
        # Remove zero-width characters
        for pattern in self._unicode_patterns:
            text = pattern.sub("", text)
        # Truncate to max length
        return text[:self.max_input_length]

    def get_security_report(self) -> Dict:
        """Generate security audit report"""
        total = len(self.audit_log)
        blocked = sum(
            1 for r in self.audit_log
            if r.level == ThreatLevel.BLOCKED
        )
        suspicious = sum(
            1 for r in self.audit_log
            if r.level == ThreatLevel.SUSPICIOUS
        )
        return {
            "total_requests": total,
            "blocked": blocked,
            "suspicious": suspicious,
            "block_rate": f"{blocked/total:.2%}" if total else "N/A",
            "top_threats": self._get_top_threats(),
        }

    def _get_top_threats(self) -> List[str]:
        from collections import Counter
        all_threats = []
        for r in self.audit_log:
            all_threats.extend(r.threats_detected)
        return [t for t, _ in Counter(all_threats).most_common(5)]

## 기업 내 생성형 AI 거버넌스 프레임워크

### 도입 전 평가 체계

### 사용 정책 수립

### RAG 기반 사내 시스템 거버넌스

**RAG 기반 생성형 AI 거버넌스 파이프라인**

In [ ]:
from dataclasses import dataclass
from typing import List, Optional, Dict
from enum import Enum
from datetime import datetime

class ContentRisk(Enum):
    SAFE = "Safe"
    NEEDS_REVIEW = "Needs Review"
    BLOCKED = "Blocked"

@dataclass
class RAGResponse:
    answer: str
    sources: List[str]
    confidence: float
    grounded: bool  # whether answer is grounded in sources

class RAGGovernancePipeline:
    """Governance pipeline for RAG-based GenAI systems"""
    def __init__(self, confidence_threshold=0.7,
                 blocked_topics=None):
        self.confidence_threshold = confidence_threshold
        self.blocked_topics = blocked_topics or [
            "medical_diagnosis", "legal_advice", "financial_advice"
        ]
        self.audit_log = []

    def pre_process(self, user_query, user_role):
        """Input guardrail: filter queries before LLM"""
        # Step 1: PII detection
        if self._contains_pii(user_query):
            return ContentRisk.BLOCKED, "PII detected in query"

        # Step 2: Topic filtering
        for topic in self.blocked_topics:
            if topic in user_query.lower():
                return ContentRisk.BLOCKED, f"Blocked topic: {topic}"

        # Step 3: Authorization check
        if not self._check_access(user_role, user_query):
            return ContentRisk.BLOCKED, "Insufficient permissions"

        return ContentRisk.SAFE, "Query approved"

    def post_process(self, response: RAGResponse):
        """Output guardrail: validate before delivery"""
        issues = []

        # Step 1: Groundedness check
        if not response.grounded:
            issues.append("Answer not grounded in retrieved sources")

        # Step 2: Confidence threshold
        if response.confidence < self.confidence_threshold:
            issues.append("Low confidence - human review recommended")

        # Step 3: Source attribution
        if not response.sources:
            issues.append("No source attribution provided")

        # Determine risk level
        if issues:
            risk = ContentRisk.NEEDS_REVIEW
        else:
            risk = ContentRisk.SAFE

        # Audit logging
        self.audit_log.append({
            "timestamp": datetime.now().isoformat(),
            "confidence": response.confidence,
            "grounded": response.grounded,
            "risk": risk.value,
            "issues": issues,
        })

        return risk, issues

    def _contains_pii(self, text):
        # Simplified PII detection logic
        import re
        patterns = [
            r'\d{6}-\d{7}',     # Korean resident ID
            r'\d{3}-\d{4}-\d{4}', # Phone number
        ]
        return any(re.search(p, text) for p in patterns)

    def _check_access(self, role, query):
        return True  # Implement RBAC logic

## LLM 평가 프레임워크

### 벤치마크 기반 평가

### 인간 평가(Human Evaluation)

### 자동 평가(LLM-as-Judge)

## 파인튜닝 거버넌스

### RLHF와 안전 정렬

### DPO(Direct Preference Optimization)

## 에이전트 AI 거버넌스

### 도구 사용 제어

### 권한 관리

### 행동 감사

## 비용 최적화 거버넌스

### 토큰 관리

### 캐싱 전략

### 모델 선택 전략

## 기술적 안전장치

### 입출력 가드레일

### 모델 평가와 레드팀

### 모니터링과 인시던트 대응

## LLM 거버넌스 종합 대시보드

**LLM 거버넌스 종합 대시보드**

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from datetime import datetime, timedelta
from enum import Enum
import json

class HealthStatus(Enum):
    HEALTHY = "healthy"
    WARNING = "warning"
    CRITICAL = "critical"

@dataclass
class ModelMetrics:
    model_id: str
    hallucination_rate: float
    guardrail_trigger_rate: float
    avg_response_time_ms: float
    daily_token_usage: int
    daily_cost_usd: float
    user_satisfaction_score: float
    injection_attempts_blocked: int
    grounding_score: float

@dataclass
class GovernanceAlert:
    severity: str
    message: str
    model_id: str
    timestamp: str
    resolved: bool = False

class LLMGovernanceDashboard:
    """Comprehensive LLM Governance Monitoring Dashboard"""

    # Governance thresholds
    THRESHOLDS = {
        "hallucination_rate": {"warning": 0.05, "critical": 0.10},
        "guardrail_trigger_rate": {"warning": 0.15, "critical": 0.30},
        "avg_response_time_ms": {"warning": 3000, "critical": 10000},
        "user_satisfaction_score": {"warning": 0.7, "critical": 0.5},
        "grounding_score": {"warning": 0.8, "critical": 0.6},
        "daily_cost_usd": {"warning": 500, "critical": 1000},
    }

    def __init__(self):
        self.models: Dict[str, List[ModelMetrics]] = {}
        self.alerts: List[GovernanceAlert] = []
        self.compliance_checks: Dict[str, bool] = {}

    def ingest_metrics(self, metrics: ModelMetrics):
        """Ingest model metrics and evaluate governance health"""
        if metrics.model_id not in self.models:
            self.models[metrics.model_id] = []
        self.models[metrics.model_id].append(metrics)

        # Evaluate thresholds and generate alerts
        self._evaluate_thresholds(metrics)

    def _evaluate_thresholds(self, metrics: ModelMetrics):
        """Check metrics against governance thresholds"""
        checks = {
            "hallucination_rate": metrics.hallucination_rate,
            "guardrail_trigger_rate": metrics.guardrail_trigger_rate,
            "avg_response_time_ms": metrics.avg_response_time_ms,
            "user_satisfaction_score": metrics.user_satisfaction_score,
            "grounding_score": metrics.grounding_score,
            "daily_cost_usd": metrics.daily_cost_usd,
        }
        # For satisfaction and grounding, lower is worse
        inverted = {"user_satisfaction_score", "grounding_score"}

        for metric_name, value in checks.items():
            thresholds = self.THRESHOLDS[metric_name]
            if metric_name in inverted:
                if value < thresholds["critical"]:
                    self._create_alert(
                        "CRITICAL", metric_name, value, metrics.model_id
                    )
                elif value < thresholds["warning"]:
                    self._create_alert(
                        "WARNING", metric_name, value, metrics.model_id
                    )
            else:
                if value > thresholds["critical"]:
                    self._create_alert(
                        "CRITICAL", metric_name, value, metrics.model_id
                    )
                elif value > thresholds["warning"]:
                    self._create_alert(
                        "WARNING", metric_name, value, metrics.model_id
                    )

    def _create_alert(self, severity, metric, value, model_id):
        self.alerts.append(GovernanceAlert(
            severity=severity,
            message=f"{metric}: {value} exceeds threshold",
            model_id=model_id,
            timestamp=datetime.now().isoformat(),
        ))

    def get_overall_health(self) -> Dict:
        """Calculate overall governance health status"""
        if not self.models:
            return {"status": "NO_DATA"}

        active_critical = sum(
            1 for a in self.alerts
            if a.severity == "CRITICAL" and not a.resolved
        )
        active_warnings = sum(
            1 for a in self.alerts
            if a.severity == "WARNING" and not a.resolved
        )

        if active_critical > 0:
            status = HealthStatus.CRITICAL
        elif active_warnings > 0:
            status = HealthStatus.WARNING
        else:
            status = HealthStatus.HEALTHY

        return {
            "status": status.value,
            "active_critical_alerts": active_critical,
            "active_warnings": active_warnings,
            "models_monitored": len(self.models),
            "total_alerts_24h": len([
                a for a in self.alerts
                if a.timestamp > (
                    datetime.now() - timedelta(hours=24)
                ).isoformat()
            ]),
        }

    def generate_compliance_report(self) -> Dict:
        """Generate compliance report for regulators"""
        report = {
            "report_date": datetime.now().isoformat(),
            "models": {},
        }
        for model_id, metrics_list in self.models.items():
            if not metrics_list:
                continue
            latest = metrics_list[-1]
            report["models"][model_id] = {
                "hallucination_rate": f"{latest.hallucination_rate:.2%}",
                "grounding_score": f"{latest.grounding_score:.2%}",
                "safety_guardrails_active": True,
                "injection_attempts_blocked":
                    latest.injection_attempts_blocked,
                "user_satisfaction": f"{latest.user_satisfaction_score:.2%}",
                "cost_efficiency": {
                    "daily_cost": f"${latest.daily_cost_usd:.2f}",
                    "tokens_per_dollar":
                        latest.daily_token_usage / max(
                            latest.daily_cost_usd, 0.01
                        ),
                },
            }
        return report

    def get_trend_analysis(self, model_id: str,
                           metric: str, days: int = 30) -> Dict:
        """Analyze metric trends over time"""
        if model_id not in self.models:
            return {"error": "Model not found"}
        history = self.models[model_id][-days:]
        values = [getattr(m, metric, None) for m in history]
        values = [v for v in values if v is not None]
        if not values:
            return {"error": "No data for metric"}
        return {
            "metric": metric, "period_days": days,
            "current": values[-1],
            "average": sum(values) / len(values),
            "min": min(values), "max": max(values),
            "trend": "improving" if len(values) > 1
                     and values[-1] < values[0] else "stable",
        }

## 생성형 AI 데이터 거버넌스

### 학습 데이터 거버넌스

### 사용자 입력 데이터 거버넌스

### 생성 출력 데이터 거버넌스

### 멀티모달 생성형 AI 거버넌스

## 규제 대응

### 글로벌 생성형 AI 규제 비교

### EU AI Act의 생성형 AI 조항

### 국내 규제 동향

## 구현 사례 연구

### 본 장 요약